## Creating the modeling

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.data import load_data, clean_data, split_data
from src.features import build_preprocessor

df = clean_data(load_data())
X_train,X_test,y_train,y_test = split_data(df)
print(X_train.shape,X_test.shape)

(5634, 19) (1409, 19)


In [8]:
import warnings
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

def evaluate (name,model):
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    row = {m: scores[f"test_{m}"].mean().round(3) for m in scoring}
    return pd.Series(row, name=name)


baseline = Pipeline([
    ("prep", build_preprocessor()),
    ("model", DummyClassifier(strategy="most_frequent")),
])
logreg = Pipeline([
    ("prep", build_preprocessor()),
    ("model", LogisticRegression(max_iter=1000)),
])

results = pd.DataFrame([
    evaluate("Baseline (always 'stay')", baseline),
    evaluate("Logistic Regression", logreg),
])
print(results)

                          accuracy  precision  recall     f1  roc_auc
Baseline (always 'stay')     0.735      0.000   0.000  0.000    0.500
Logistic Regression          0.802      0.653   0.543  0.592    0.846


## Trying stronger models

In [9]:
import warnings
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

logreg_balanced = Pipeline([
    ("prep", build_preprocessor()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
rf = Pipeline([
    ("prep", build_preprocessor()),
    ("model", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1)),
])
xgb = Pipeline([
    ("prep", build_preprocessor()),
    ("model", XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=3,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", random_state=42)),
])

results = pd.DataFrame([
    evaluate("Baseline (always 'stay')", baseline),
    evaluate("Logistic Regression", logreg),
    evaluate("Logistic Reg. (balanced)", logreg_balanced),
    evaluate("Random Forest", rf),
    evaluate("XGBoost", xgb),
])
print(results)


                          accuracy  precision  recall     f1  roc_auc
Baseline (always 'stay')     0.735      0.000   0.000  0.000    0.500
Logistic Regression          0.802      0.653   0.543  0.592    0.846
Logistic Reg. (balanced)     0.748      0.517   0.801  0.628    0.846
Random Forest                0.802      0.672   0.496  0.570    0.845
XGBoost                      0.803      0.662   0.529  0.588    0.847
